# 06 — Temporal trends (inter-surgical delay, H4)

**Notebook id**: `06_temporal` → vault `03.7-tendances-temporelles.md`

**H4**: cyclops patients are re-operated **sooner** (shorter `inter_surgery_d`).

- **DAG / causal status** (point F): the delay is a **mediator** *downstream of group* (Group → delay → progression, and Group → progression). It is therefore the **outcome** of H4 — it is **not** a confounder of the PF effect, and the primary model does **not** adjust for it. Conditioning on it (as a sensitivity / mediation analysis) *strengthens* the PF effect.
- **Frequentist**: Mann–Whitney on `inter_surgery_d` (cyclops vs meniscus).
- **Time-at-risk falsification**: the delay-**adjusted** worsened-PF OR (`tf.firth_or(covariates=('inter_surgery_d',))`) + the correlation ρ(delay, worsened_pf). If the delay were the *biological* driver, adjusting for it would collapse the OR and ρ would be strongly negative; instead the OR is preserved and ρ ≈ 0 — consistent with **time-at-risk**, not a confounder/mediator that explains the PF effect away.
- **M4**: Weibull AFT (lifelines MLE).
- **M5**: per-group LogNormal (NUTS via nutpie) — sampling cell is runnable but not executed here.
- **Figure**: ECDF + fitted LogNormal per group with median annotations.

> **Anomaly note**: the date anomalies flagged in 00 (#9 trauma-date drift; #25/#38 negative trauma→surgery) corrupt *age / trauma-to-surgery*, **not** `inter_surgery_d` — the latter is the **difference of two `date_chir`** (S2 − S1), neither of which is hygienised or negative here. H4 is therefore robust to those source-date issues.

In [ ]:
# --- Setup (idempotent, fresh-kernel reproducible) ---
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd

from constants import (
    RANDOM_SEED, SITES, SITES_PF, SITES_FT, SITES_BINARY, SITES_ORDINAL,
    GROUPS, BLOCKS, N_TOTAL, N_TOTAL_ANALYSABLE, N_MENISCUS, N_CYCLOPS,
    SCORE_MAX, SCORE_MAX_COLLAPSED,
)
import loaders
import preprocessing as pp
import tests_freq as tf
import reporting as rpt
import bayes_models as bm
import viz

np.random.seed(RANDOM_SEED)
viz.set_pub_style()


In [ ]:
# --- Load & preprocess (canonical pipeline) ---
df = loaders.load_combined()
df = pp.apply_date_hygiene(df)      # composite-key (group, anonyme) date hygiene
df = pp.add_derived(df)            # lesion_pf/ft, female, deltas, worsened_pf, ...
wide = pp.to_wide(df)             # one row per patient (group, anonyme)
patient = pp.to_patient(df)       # static covariates per patient

# Patient-level covariates joined onto the wide outcomes (for H3 / sensitivity).
_cov = [c for c in ['group','anonyme','female','sexe','pivot_pivot_contact',
                    'travail_physique','tabac','age_at_trauma','imc','taille','poids']
        if c in patient.columns]
merged = wide.merge(patient[_cov], on=['group','anonyme'], how='left')

print('long:', df.shape, '| wide:', wide.shape, '| patient:', patient.shape,
      '| merged:', merged.shape)
# Composite-key sentinel: 19 Anonyme ids are reused across the two sheets.
assert (df.groupby(['group','anonyme']).size() == 2).all(), 'composite key broken'


## 1. DAG (time-at-risk, not a confounder)

```
        Group (cyclops vs meniscus)
        /                        \
       v                          v
  inter_surgery_d  ----------->  PF progression
   (time-at-risk / mediator)      (Δ lesion_pf)
```
The delay is **time-at-risk** sitting *downstream* of group, not a confounder. The primary PF estimand is the **total** effect of group; we do **not** condition on the delay in the primary model (that would over-adjust a mediator). H4 treats the delay as an **outcome**, and §3b uses it only as a falsification check.

## 2. Distribution of the inter-surgical delay

In [ ]:
isd = wide[['group','inter_surgery_d']].dropna()
print(isd.groupby('group')['inter_surgery_d'].describe())
print()
print('Medians (days):')
print(isd.groupby('group')['inter_surgery_d'].median())


## 3. Mann–Whitney on `inter_surgery_d` (H4)

In [ ]:
c = isd.loc[isd.group=='cyclops','inter_surgery_d'].astype(float).values
m = isd.loc[isd.group=='meniscus','inter_surgery_d'].astype(float).values
r = tf.mwu_with_effects(c, m, n_boot=10000, seed=RANDOM_SEED)
print(f"inter_surgery_d: Cliff delta = {r['cliffs_delta']:+.3f} "
      f"({r['cliffs_delta_magnitude']}), MWU p = {r['pvalue']:.5f}")
print(f"BCa 95% CI on delta = [{r['delta_ci_lo']:+.3f}, {r['delta_ci_hi']:+.3f}]")


## 3b. Time-at-risk falsification — delay-adjusted OR + ρ(delay, worsened-PF)

Two checks that the delay is **time-at-risk**, not the biological cause of the PF effect:

1. **Delay-adjusted Firth OR** — `tf.firth_or(worsened_pf, covariates=('inter_surgery_d',))`. If the delay *mediated* the effect, conditioning on it would shrink the OR toward 1. Instead the OR is **kept** (if anything inflated, because more time-at-risk in cyclops is part of the causal path, not a confound) — so the PF effect is not explained away by the delay.
2. **Falsification correlation** ρ(`inter_surgery_d`, `worsened_pf`) within the analysable cohort: a *biological-mediator* story predicts a clear negative ρ; **ρ ≈ 0** is what time-at-risk predicts here.

In [ ]:
# Delay-adjusted worsened-PF OR (Firth; merged carries inter_surgery_d + covariates).
or_delay = tf.firth_or(merged, outcome_col='worsened_pf',
                       covariates=('inter_surgery_d',))
print('Delay-ADJUSTED Firth OR for worsened_pf (cyclops vs meniscus):')
print(f"  OR = {or_delay['odds_ratio']:.2f}  "
      f"[{or_delay['or_ci_lo']:.2f}, {or_delay['or_ci_hi']:.2f}]  "
      f"p = {or_delay['p']:.4f}  n = {or_delay['n']}  method={or_delay['method']}")
print('  (the PF effect SURVIVES adjustment for the delay => not delay-mediated;')
print('   the delay is time-at-risk on the causal path, not a confounder.)')
print()
# Falsification correlation: delay vs worsened_pf should be ~0 (not a mediator).
fal = merged[['inter_surgery_d','worsened_pf']].apply(pd.to_numeric, errors='coerce').dropna()
rho = tf.spearman_bca(fal['inter_surgery_d'].values, fal['worsened_pf'].values,
                      n_boot=2000, seed=RANDOM_SEED)
print('Falsification correlation rho(inter_surgery_d, worsened_pf):')
print(f"  rho = {rho['rho']:+.3f}  CI [{rho['ci_lo']:+.3f}, {rho['ci_hi']:+.3f}]  "
      f"p = {rho['pvalue']:.3f}  n = {rho['n']}")
print('  (rho ~ 0 => no monotone delay->worsening link; consistent with time-at-risk.)')


## 4. M4 — Weibull AFT on `inter_surgery_d` (lifelines MLE)

Accelerated-failure-time on the delay (no censoring). A positive group coefficient ⇒ longer delay; cyclops should be shorter.

In [ ]:
aft_cov = [c for c in ['group','imc','pivot_pivot_contact'] if c in merged.columns]
m4 = bm.fit_m4_weibull_aft(merged, duration_col='inter_surgery_d', covariates=aft_cov)
print(m4['summary'])


## 5. M5 — per-group LogNormal on `inter_surgery_d` (runnable sampling cell)

This cell **is runnable** (uncommented). Execute it yourself to sample (NUTS via nutpie). Posterior median delay per group = exp(μ_group).

In [ ]:
# Fit M5 LogNormal on the inter-surgical delay (H4 outcome). Runs NUTS via nutpie.
idata_m5 = bm.fit_m5_lognormal(wide, var='inter_surgery_d', nuts_sampler='nutpie')
diag5 = bm.check_convergence(idata_m5)
print('M5 convergence:', diag5)
summ5 = rpt.summary_bayes(idata_m5, var_names=['mu','sigma'], hdi_prob=0.94)
print(summ5)
medians = {g: float(np.exp(idata_m5.posterior['mu'].sel(group=g).values.mean()))
           for g in idata_m5.posterior['mu'].coords['group'].values}
print('Posterior median delay by group (days):',
      {g: round(v,1) for g, v in medians.items()})
# Optional: cache for make_figures.py.
# idata_m5.to_netcdf('../results/idata_m5.nc')


## 6. Figure — delay ECDF + fitted LogNormal per group

Empirical ECDF overlaid with the fitted LogNormal (M5 posterior-mean μ/σ when `idata_m5` is in scope, else a per-group MLE), with medians and the mediator caption.

In [ ]:
medians = {g: float(isd.loc[isd.group==g,'inter_surgery_d'].median()) for g in GROUPS}
_idata5 = idata_m5 if 'idata_m5' in dir() else None
fig = viz.delay_ecdf_fit(wide, idata=_idata5, value_col='inter_surgery_d',
                         medians=medians)
fig


## Sanity asserts

In [ ]:
assert wide['inter_surgery_d'].notna().sum() >= 60
assert isd.groupby('group')['inter_surgery_d'].median()['cyclops'] <        isd.groupby('group')['inter_surgery_d'].median()['meniscus'],        'cyclops should be re-operated sooner (H4)'
print('Temporal / H4 asserts passed.')
